# Polylines and streamlines

**Geometry types:** `polyline` · `streamline`

Ordered vertex sequences. Use `streamline` for tractography data (adds
step-size and propagation metadata). All read/write functions are identical
for both types.

1. Generate synthetic tractography streamlines
2. Write with groups and per-object attributes
3. Open lazily and inspect metadata
4. Read all / by object ID / by group
5. Spatial bounding-box query
6. Build multi-resolution pyramid with object sparsity
7. Read at coarser levels
8. Export to TRK


In [1]:
import numpy as np, tempfile, os
from zarr_vectors.lazy import open_zvr
from zarr_vectors.types.polylines import write_polylines, read_polylines
from zarr_vectors.validate import validate

_tmpdir = tempfile.mkdtemp(prefix="zvf_examples_")
STORE = os.path.join(_tmpdir, "tracts.zarrvectors")
print("Store path:", STORE)

Store path: /tmp/zvf_examples_irtjrt76/tracts.zarrvectors


## 1 · Generate synthetic tractography streamlines

In [2]:
rng = np.random.default_rng(1)
N = 2_000   # 2000 streamlines

streamlines = []
for i in range(N):
    n_verts = rng.integers(30, 100)
    sl = rng.normal(0, 15, (n_verts, 3)).cumsum(axis=0).astype(np.float32)
    # Centre streamlines around origin in a ~300 mm white-matter volume
    sl += rng.uniform(-100, 100, 3).astype(np.float32)
    streamlines.append(sl)

# Per-streamline attributes (object_attributes — one value per streamline)
lengths  = np.array([np.sum(np.linalg.norm(np.diff(s, axis=0), axis=1))
                     for s in streamlines], dtype=np.float32)
mean_fa  = rng.uniform(0.2, 0.7, N).astype(np.float32)

# Per-vertex FA — list of arrays, one array per streamline (vertex_attributes)
per_vertex_fa = [rng.uniform(0.1, 0.9, len(s)).astype(np.float32)
                 for s in streamlines]

# Group into 3 bundles. `groups` keys must be ints; values are object IDs.
bundle_size = N // 3
groups = {
    0: list(range(bundle_size)),
    1: list(range(bundle_size, 2 * bundle_size)),
    2: list(range(2 * bundle_size, N)),
}
group_names = np.array(["CST", "AF", "UF"], dtype=object)

print(f"Generated {N} streamlines")
print(f"Length range: {lengths.min():.1f} – {lengths.max():.1f} mm")
print(f"Groups: {[(group_names[gid], len(ids)) for gid, ids in groups.items()]}")

Generated 2000 streamlines
Length range: 560.5 – 2568.6 mm
Groups: [('CST', 666), ('AF', 666), ('UF', 668)]


## 2 · Write with groups and attributes

In [3]:
write_polylines(
    STORE,
    streamlines,
    chunk_shape=(100.0, 100.0, 100.0),
    bin_shape=(25.0, 25.0, 25.0),
    geometry_type="streamline",
    groups=groups,
    object_attributes={
        "length":  lengths,
        "mean_fa": mean_fa,
    },
    vertex_attributes={"fa": per_vertex_fa},
)
print("Write complete.")

Write complete.


## 3 · Open lazily and inspect metadata

In [4]:
store = open_zvr(STORE)
print(f"geometry_types : {store.geometry_types}")
print(f"ndim           : {store.ndim}")
print(f"chunk_shape    : {store.chunk_shape}")
print(f"n_objects      : {store[0].object_index.object_count:,}   (= {N} streamlines)")
print(f"vertex_count   : {store[0].vertex_count:,}")
lo, hi = np.array(store.bounds[0]), np.array(store.bounds[1])
print(f"bounds         :\n  lo={lo.round(1)}\n  hi={hi.round(1)}")

geometry_types : ['streamline']
ndim           : 3
chunk_shape    : (100.0, 100.0, 100.0)
n_objects      : 2,000   (= 2000 streamlines)
vertex_count   : 129,084
bounds         :
  lo=[-451.4 -519.2 -548.1]
  hi=[560.6 504.9 506.6]


## 4a · Read all streamlines

In [5]:
result_all = read_polylines(STORE)
print(f"polyline_count : {result_all['polyline_count']:,}")
print(f"vertex_count   : {result_all['vertex_count']:,}")

# Each polyline is a LIST of segment arrays (one per chunk it crosses).
# Concatenate to get the full path:
sl0 = np.concatenate(result_all['polylines'][0])
sl1 = np.concatenate(result_all['polylines'][1])
print(f"\nStreamline 0: {sl0.shape[0]} vertices total ({len(result_all['polylines'][0])} segments)")
print(f"Streamline 1: {sl1.shape[0]} vertices total ({len(result_all['polylines'][1])} segments)")

polyline_count : 2,000
vertex_count   : 129,084

Streamline 0: 63 vertices total (48 segments)
Streamline 1: 65 vertices total (52 segments)


## 4b · Read by object ID

In [6]:
result_id = read_polylines(STORE, object_ids=[0, 42, 999])
print(f"Read {result_id['polyline_count']} streamlines by ID")
sl42 = np.concatenate(result_id['polylines'][1])
print(f"Streamline 42 has {sl42.shape[0]} vertices")

Read 3 streamlines by ID
Streamline 42 has 49 vertices


## 4c · Read by group

In [7]:
# `groups` keys are integers; here we mapped 0/1/2 → CST/AF/UF
for gid, name in enumerate(group_names):
    r = read_polylines(STORE, group_ids=[int(gid)])
    print(f"Group {gid} ({name}): {r['polyline_count']:,} streamlines")

Group 0 (CST): 666 streamlines
Group 1 (AF): 666 streamlines
Group 2 (UF): 668 streamlines


## 4d · Read per-object attributes without loading geometry

In [8]:
from zarr_vectors.core.store import open_store, get_resolution_level
from zarr_vectors.core.arrays import read_object_attributes

root = open_store(STORE)
level0 = get_resolution_level(root, 0)
lengths_stored = read_object_attributes(level0, "length")
fa_stored      = read_object_attributes(level0, "mean_fa")

print(f"mean_fa range: [{fa_stored.min():.3f}, {fa_stored.max():.3f}]")
print(f"length  range: [{lengths_stored.min():.1f}, {lengths_stored.max():.1f}] mm")

# Select streamlines with high FA and length > 50 mm
selected = np.where((fa_stored > 0.55) & (lengths_stored > 50))[0]
print(f"\nHigh-FA long streamlines: {len(selected)}")
result_sel = read_polylines(STORE, object_ids=selected[:10].tolist())
print(f"Loaded {result_sel['polyline_count']} (first 10 of selection)")

mean_fa range: [0.201, 0.700]
length  range: [560.5, 2568.6] mm

High-FA long streamlines: 622
Loaded 10 (first 10 of selection)


## 5 · Spatial bounding-box query

In [9]:
lo = np.array([-30.0, -30.0, -30.0])
hi = np.array([ 30.0,  30.0,  30.0])

bbox_result = read_polylines(STORE, bbox=(lo, hi))
print(f"Streamlines passing through ±30 mm cube: {bbox_result['polyline_count']:,}")
print("(Full streamline geometry is returned, not clipped to bbox)")


Streamlines passing through ±30 mm cube: 1,907
(Full streamline geometry is returned, not clipped to bbox)


## 6 · Build multi-resolution pyramid with object sparsity

In [10]:
from zarr_vectors.multiresolution.coarsen import build_pyramid

build_pyramid(
    STORE,
    level_configs=[
        {"bin_ratio": (2, 2, 2), "object_sparsity": 1.0,
         "sparsity_strategy": "spatial_coverage"},
        {"bin_ratio": (4, 4, 4), "object_sparsity": 0.25,
         "sparsity_strategy": "spatial_coverage"},
    ],
    agg_mode="mean",
)
print("Pyramid built.")

Pyramid built.


## 7 · Read at coarser levels

In [11]:
store._level_list = None  # invalidate cached level list after pyramid build
for lv in store.levels:
    n_obj  = store[lv].object_index.object_count
    n_vert = store[lv].vertex_count
    bs     = store[lv].bin_shape
    print(f"Level {lv}: {n_obj:>5,} objects  {n_vert:>8,} vertices  bin_shape={bs}")

print()
coarse = read_polylines(STORE, level=2)
total = sum(np.concatenate(p).shape[0] for p in coarse['polylines'] if p)
print(f"Level 2: {coarse['polyline_count']} polylines, "
      f"avg {total / max(1, coarse['polyline_count']):.1f} vertices/polyline")

Level 0: 2,000 objects   129,084 vertices  bin_shape=None
Level 1:     0 objects     2,365 vertices  bin_shape=(50.0, 50.0, 50.0)
Level 2:     0 objects       113 vertices  bin_shape=(100.0, 100.0, 100.0)

Level 2: 0 polylines, avg 0.0 vertices/polyline


## 8 · Validate

In [12]:
result_v = validate(STORE, level=3)
print(result_v.summary())


Level 3 validation: PASS
  36 passed, 0 warnings, 0 errors


## Summary

Each polyline is split across the chunks it passes through; `read_polylines`
returns one list of segment arrays per polyline. Concatenate to get the
complete path.

| Step | API |
|------|-----|
| Write | `write_polylines(path, polylines, vertex_attributes, object_attributes, groups)` |
| Read all | `read_polylines(path)` → `polylines[i]` is a list of segment arrays |
| Read by ID | `read_polylines(path, object_ids=[...])` |
| Read by group | `read_polylines(path, group_ids=[int, ...])` |
| Bbox query | `read_polylines(path, bbox=(lo, hi))` |
| Object attributes | `read_object_attributes(level_group, name)` |
| Pyramid | `build_pyramid(path, level_configs, agg_mode="mean")` |